<a href="https://colab.research.google.com/github/zaku2590/classGCI/blob/main/comp2%E3%83%81%E3%83%A5%E3%83%BC%E3%83%8B%E3%83%B3%E3%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna
!pip install catboost xgboost
!pip uninstall xgboost -y
!pip install xgboost --upgrade


In [ ]:
# モジュールのインポート
import optuna
import lightgbm as lgb
import numpy as np  # 数値計算や配列操作を行うためのライブラリ
import pandas as pd  # 表形式のデータを扱うためのライブラリ
import matplotlib.pyplot as plt  # データ可視化のための基本的なグラフ描画ライブラリ
import seaborn as sns  # 高機能な統計グラフを描画するライブラリ
from sklearn.preprocessing import LabelEncoder  # カテゴリ変数を数値に変換するエンコーダ
from sklearn.ensemble import RandomForestClassifier  # ランダムフォレストによる分類器
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold ,cross_val_score # 層化K分割交差検証を行うクラス
from sklearn.metrics import roc_auc_score  # ROC AUCスコアを計算する評価指標
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
PATH = '/content/'

train = pd.read_csv(PATH + 'train.csv')
test = pd.read_csv(PATH + 'test.csv')


In [ ]:
# 使わない列の削除
train = train.drop(columns=["Id"])
test = test.drop(columns=["Id"])

cols_to_important = ['Age', 'Agility_3cone', 'Shuttle']

for data in cols_to_important:

    train[data + "_was_missing"] = train[data].isnull().astype(int)
    test[data + "_was_missing"] = test[data].isnull().astype(int)

# 平均で補完する対象の列
cols_to_fill = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                'Broad_Jump', 'Agility_3cone', 'Shuttle']

# positionTypeで平均を埋める
for col in cols_to_fill:

    group_mean = train.groupby("Position_Type")[col].mean()
    train[col] = train[col].fillna(train["Position_Type"].map(group_mean))
    test[col] = test[col].fillna(test["Position_Type"].map(group_mean))

for col in ['Vertical_Jump', 'Sprint_40yd', 'Agility_3cone']:
    # 学習データ側で Position ごとの mean/std を事前に計算
    pos_stats = train.groupby("Position")[col].agg(["mean", "std"]).rename(columns={"mean": "mean_val", "std": "std_val"})

    # train にマージして計算
    train = train.merge(pos_stats, left_on="Position", right_index=True, how="left")
    train[f"{col}_z_by_pos"] = ((train[col] - train["mean_val"]) / train["std_val"]).fillna(0)
    train.drop(columns=["mean_val", "std_val"], inplace=True)

    # test も同様に train の mean/std を使用
    test = test.merge(pos_stats, left_on="Position", right_index=True, how="left")
    test[f"{col}_z_by_pos"] = ((test[col] - test["mean_val"]) / test["std_val"]).fillna(0)
    test.drop(columns=["mean_val", "std_val"], inplace=True)

train["BMI"] = train["Weight"] / (train["Height"] ** 2)
test["BMI"] = test["Weight"] / (test["Height"] ** 2)


train["Power_Index"] = train["Vertical_Jump"] * train["Weight"]
test["Power_Index"] = test["Vertical_Jump"] * test["Weight"]

train["Jump_per_kg"] = train["Vertical_Jump"] / train["Weight"]
test["Jump_per_kg"] = test["Vertical_Jump"] / test["Weight"]

train["Strength_per_kg"] = train["Bench_Press_Reps"] / train["Weight"]
test["Strength_per_kg"] = test["Bench_Press_Reps"] / test["Weight"]

# 総出力（パワー的な指標）
train["Total_Power"] = train["Bench_Press_Reps"] * train["Weight"]
test["Total_Power"] = test["Bench_Press_Reps"] * test["Weight"]

# 爆発力（ジャンプの距離 ÷ 走力）
train["Explosiveness_Index"] = train["Broad_Jump"] / train["Sprint_40yd"]
test["Explosiveness_Index"] = test["Broad_Jump"] / test["Sprint_40yd"]

train["Power_Ratio"] = train["Power_Index"] / train["BMI"]
test["Power_Ratio"] = test["Power_Index"] / test["BMI"]

# Agility_x_Strength
train["Agility_x_Strength"] = train["Agility_3cone"] * train["Strength_per_kg"]
test["Agility_x_Strength"] = test["Agility_3cone"] * test["Strength_per_kg"]

# Power_Index / Sprint_40yd: パワーをどれだけ速く出せるか
train["Power_to_Speed"] = train["Power_Index"] / train["Sprint_40yd"]
test["Power_to_Speed"] = test["Power_Index"] / test["Sprint_40yd"]

# BMI * Explosiveness_Index: 体格に対しての爆発力
train["BMI_x_Explosiveness"] = train["BMI"] * train["Explosiveness_Index"]
test["BMI_x_Explosiveness"] = test["BMI"] * test["Explosiveness_Index"]

# Strength_per_kg * Explosiveness_Index: 筋力と爆発力の掛け合わせ
train["Strength_x_Explosiveness"] = train["Strength_per_kg"] * train["Explosiveness_Index"]
test["Strength_x_Explosiveness"] = test["Strength_per_kg"] * test["Explosiveness_Index"]

# Sprint_40yd / Bench_Press_Reps: スピードと筋力のバランス（少ないほど優秀）
train["Speed_to_Strength_Ratio"] = train["Sprint_40yd"] / (train["Bench_Press_Reps"] + 1e-5)
test["Speed_to_Strength_Ratio"] = test["Sprint_40yd"] / (test["Bench_Press_Reps"] + 1e-5)

# Height * Agility_3cone: 身長と俊敏性の交差
train["Height_x_Agility"] = train["Height"] * train["Agility_3cone"]
test["Height_x_Agility"] = test["Height"] * test["Agility_3cone"]

# Strength_to_Speed = Bench_Press_Reps / Sprint_40yd
# train["Strength_to_Speed"] = train["Bench_Press_Reps"] / train["Sprint_40yd"]
# test["Strength_to_Speed"] = test["Bench_Press_Reps"] / test["Sprint_40yd"]

# # Height_to_Weight = Height / Weight
# train["Height_to_Weight"] = train["Height"] / train["Weight"]
# test["Height_to_Weight"] = test["Height"] / test["Weight"]

# Speed_x_Explosive
# train["Speed_x_Explosive"] = train["Sprint_40yd"] * train["Explosiveness_Index"]
# test["Speed_x_Explosive"] = test["Sprint_40yd"] * test["Explosiveness_Index"]

# BMI_x_Speed
# train["BMI_x_Speed"] = train["BMI"] * train["Sprint_40yd"]
# test["BMI_x_Speed"] = test["BMI"] * test["Sprint_40yd"]

# numeric_cols = train.select_dtypes(include=[np.number]).drop(columns=["Drafted"]).columns
# kmeans = KMeans(n_clusters=9, random_state=42, n_init=10)
# train["Cluster"] = kmeans.fit_predict(train[numeric_cols])
# test["Cluster"] = kmeans.predict(test[numeric_cols])

# # Cluster列をターゲットエンコーディング
# target_mean = train.groupby("Cluster")["Drafted"].mean()
# train["Cluster_TE"] = train["Cluster"].map(target_mean)
# test["Cluster_TE"] = test["Cluster"].map(target_mean)
# train = train.drop(columns=["Cluster"])
# test = test.drop(columns=["Cluster"])

all_data = pd.concat([train[["School"]], test[["School"]]])

# 各Schoolの出現回数をカウント
school_counts = all_data["School"].value_counts().to_dict()

# train にマップ
train["School_Count"] = train["School"].map(school_counts)

# test にマップ
test["School_Count"] = test["School"].map(school_counts)

train = train.drop(columns=["Shuttle", "Bench_Press_Reps", "School"])
test = test.drop(columns=["Shuttle", "Bench_Press_Reps", "School"])

train.head()

In [71]:
import optuna
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier, callback
from catboost import CatBoostClassifier

# =============== データ準備 ===============
X_base = train.drop(columns=["Drafted"])
y = train["Drafted"]
te_columns = ["Player_Type", "Position_Type", "Position"]

# =============== 最適化関数 ===============
def optimize_model(model_name, n_trials=30):
    def objective(trial):
        if model_name == "lgb":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 31, 1023),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "n_estimators": 1000,
                "random_state": 2025,
                "verbosity": -1
            }
        elif model_name == "xgb":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "n_estimators": 1000,
                "random_state": 2025,
                "use_label_encoder": False,
                "eval_metric": "auc",
                "verbosity": 0
            }
        elif model_name == "cat":
            params = {
                "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
                "depth": trial.suggest_int("depth", 3, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "random_strength": trial.suggest_float("random_strength", 0.0, 1.0),
                "border_count": trial.suggest_int("border_count", 32, 255),
                "iterations": 1000,
                "eval_metric": "AUC",
                "random_state": 2025,
                "verbose": 0
            }
        else:
            raise ValueError("Invalid model_name")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        aucs = []

        for train_idx, valid_idx in skf.split(X_base, y):
            X_train, X_valid = X_base.iloc[train_idx].copy(), X_base.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            # ターゲットエンコーディング
            for col in te_columns:
                te_map = y_train.groupby(X_train[col]).mean()
                X_train[f"{col}_TE"] = X_train[col].map(te_map).fillna(0)
                X_valid[f"{col}_TE"] = X_valid[col].map(te_map).fillna(0)
                X_train.drop(columns=[col], inplace=True)
                X_valid.drop(columns=[col], inplace=True)

            for df in [X_train, X_valid]:
                df.replace([np.inf, -np.inf], np.nan, inplace=True)
                df.fillna(0, inplace=True)

            if model_name == "lgb":
                model = LGBMClassifier(**params)
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_valid, y_valid)],
                    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
                )
            elif model_name == "xgb":
                model = XGBClassifier(**params)
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_valid, y_valid)],
                    callbacks=[callback.EarlyStopping(rounds=50, save_best=True)]
                )
            elif model_name == "cat":
                model = CatBoostClassifier(**params)
                model.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=50, verbose=False)

            preds = model.predict_proba(X_valid)[:, 1]
            aucs.append(roc_auc_score(y_valid, preds))

        mean_auc = np.mean(aucs)
        print(f"✅ [{model_name}] Fold AUCs: {[round(a, 4) for a in aucs]} | Mean AUC: {round(mean_auc, 4)}")
        return mean_auc

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    print(f"\n⭐ {model_name.upper()} Best Params: {study.best_params}")
    print(f"⭐ {model_name.upper()} Best AUC: {round(study.best_value, 4)}\n")
    return study

# =============== 実行 ===============
study_lgb = optimize_model("lgb", n_trials=30)
study_xgb = optimize_model("xgb", n_trials=30)
study_cat = optimize_model("cat", n_trials=30)


[I 2025-07-04 07:45:35,480] A new study created in memory with name: no-name-e4639ab7-0792-473a-be35-dd293c346ad3
[I 2025-07-04 07:45:36,256] Trial 0 finished with value: 0.8309336120448606 and parameters: {'learning_rate': 0.08847677826443003, 'num_leaves': 1022, 'max_depth': 3, 'min_child_samples': 34, 'subsample': 0.7239983826956975, 'colsample_bytree': 0.9683414916411242, 'reg_alpha': 0.0045932873664858995, 'reg_lambda': 2.312774931199805}. Best is trial 0 with value: 0.8309336120448606.


✅ [lgb] Fold AUCs: [np.float64(0.8024), np.float64(0.8504), np.float64(0.8657), np.float64(0.7944), np.float64(0.8418)] | Mean AUC: 0.8309


[I 2025-07-04 07:45:39,407] Trial 1 finished with value: 0.8285087073042158 and parameters: {'learning_rate': 0.011006989965448919, 'num_leaves': 182, 'max_depth': 6, 'min_child_samples': 57, 'subsample': 0.9088228203874984, 'colsample_bytree': 0.93643519659391, 'reg_alpha': 1.3413608062209515e-07, 'reg_lambda': 0.10372070160615858}. Best is trial 0 with value: 0.8309336120448606.


✅ [lgb] Fold AUCs: [np.float64(0.7961), np.float64(0.8548), np.float64(0.8598), np.float64(0.7911), np.float64(0.8407)] | Mean AUC: 0.8285


[I 2025-07-04 07:45:40,499] Trial 2 finished with value: 0.8307832373129662 and parameters: {'learning_rate': 0.09275242109542259, 'num_leaves': 693, 'max_depth': 3, 'min_child_samples': 32, 'subsample': 0.8372108043928098, 'colsample_bytree': 0.5854076285380089, 'reg_alpha': 1.3346894472799074e-06, 'reg_lambda': 0.6676983613960767}. Best is trial 0 with value: 0.8309336120448606.


...

[I 2025-07-04 07:48:09,820] Trial 28 finished with value: 0.8339479736066597 and parameters: {'learning_rate': 0.0024226882100194694, 'num_leaves': 916, 'max_depth': 4, 'min_child_samples': 100, 'subsample': 0.608190128872131, 'colsample_bytree': 0.5404208434324749, 'reg_alpha': 0.006040205675590339, 'reg_lambda': 0.00032129273721645996}. Best is trial 26 with value: 0.8340947329283301.


✅ [lgb] Fold AUCs: [np.float64(0.8008), np.float64(0.8636), np.float64(0.8685), np.float64(0.7904), np.float64(0.8464)] | Mean AUC: 0.8339


[I 2025-07-04 07:48:13,391] Trial 29 finished with value: 0.8299557338739174 and parameters: {'learning_rate': 0.0014176024458500728, 'num_leaves': 968, 'max_depth': 3, 'min_child_samples': 93, 'subsample': 0.6057448507472015, 'colsample_bytree': 0.5253445544513444, 'reg_alpha': 0.006719433590886781, 'reg_lambda': 0.0005337943902251666}. Best is trial 26 with value: 0.8340947329283301.
[I 2025-07-04 07:48:13,393] A new study created in memory with name: no-name-26c9d163-3fb9-401d-a40c-ad1b696f7bd7
[W 2025-07-04 07:48:13,414] Trial 0 failed with parameters: {'learning_rate': 0.001864154232358361, 'max_depth': 4, 'subsample': 0.9816915033787776, 'colsample_bytree': 0.5871662052428207, 'reg_alpha': 6.891991108814149e-07, 'reg_lambda': 0.7485705023629746} because of the following error: TypeError("XGBClassifier.fit() got an unexpected keyword argument 'callbacks'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _

✅ [lgb] Fold AUCs: [np.float64(0.8004), np.float64(0.8584), np.float64(0.8674), np.float64(0.7849), np.float64(0.8386)] | Mean AUC: 0.83

⭐ LGB Best Params: {'learning_rate': 0.004245837992110616, 'num_leaves': 559, 'max_depth': 3, 'min_child_samples': 68, 'subsample': 0.6149263830411024, 'colsample_bytree': 0.5629802512126598, 'reg_alpha': 0.007196579334743112, 'reg_lambda': 4.948611733018113e-07}
⭐ LGB Best AUC: 0.8341



TypeError: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'

In [ ]:
# # モデル訓練（全体で簡易実行）
# model = LGBMClassifier(
#     learning_rate=0.010488294829397215,
#     num_leaves=96,
#     max_depth=3,
#     min_child_samples=74,
#     subsample=0.6219322010855215,
#     colsample_bytree=0.6022862354233319,
#     reg_alpha=0.8351184813440499,
#     reg_lambda=0.6559328766031582,
#     n_estimators=1000,
#     random_state=2025,
#     verbosity=-1
# )
# model.fit(X, y)

# # 重要度取得
# importances = model.feature_importances_
# features = X.columns

# importance_df = pd.DataFrame({
#     'Feature': features,
#     'Importance': importances
# }).sort_values(by='Importance', ascending=False)

# # 可視化
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 6))
# plt.barh(importance_df['Feature'], importance_df['Importance'])
# plt.gca().invert_yaxis()
# plt.title("Feature Importances")
# plt.show()


In [ ]:
submission_template = pd.read_csv(PATH + 'sample_submission.csv')

for name, test_pred in model_test_pred_dict.items():
    submission = submission_template.copy()
    submission["Drafted"] = test_pred
    submission.to_csv(PATH + f"{name.lower()}_submission.csv", index=False)

In [ ]:
# sample_submission.csv 読み込み
submission_template = pd.read_csv(PATH + 'sample_submission.csv')

# --- 各モデルごとの提出ファイル ---
for name, test_pred in model_test_pred_dict.items():
    submission = submission_template.copy()
    submission["Drafted"] = test_pred
    submission.to_csv(PATH + f"{name.lower()}_submission.csv", index=False)

# === アンサンブル（加重平均） ===

# AUCが高い順にソート
sorted_models = sorted(model_auc_dict.items(), key=lambda x: x[1], reverse=True)
top_model, second_model, third_model = sorted_models[0][0], sorted_models[1][0], sorted_models[2][0]

print(f"\n=== Weighted Ensemble ===")
print(f"Top Model: {top_model} (weight=0.5)")
print(f"Other Models: {second_model}, {third_model} (weight=0.25 each)")

# 加重平均の予測確率を計算
ensemble_pred = (
    0.5  * model_test_pred_dict[top_model] +
    0.25 * model_test_pred_dict[second_model] +
    0.25 * model_test_pred_dict[third_model]
)

# --- アンサンブル提出ファイル ---
submission = submission_template.copy()
submission["Drafted"] = ensemble_pred

ensemble_name = "weighted_ensemble"
submission.to_csv(PATH + f"{ensemble_name.lower()}_submission.csv", index=False)

print(f"✅ Saved: {PATH}{ensemble_name.lower()}_submission.csv")
